# GROBID PDF Parsing + Pinecone Embedding

In [1]:
from langchain_community.document_loaders.parsers import GrobidParser
from langchain_community.document_loaders.generic import GenericLoader

/Users/a215482/Desktop/personal projects/rag-citation-langchain/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
try:
    parser = GrobidParser(segment_sentences=False)
    print("Parser created")
    print(parser)
except Exception as e:
    print(f"Error: {e}")


Parser created


In [3]:

loader = GenericLoader.from_filesystem(
    ".",
    glob="hope.pdf",
    suffixes=[".pdf"],
    parser=GrobidParser(segment_sentences=False)
)

In [4]:
try :
    loader.load()
except Exception as e:
    print(f"Error: {e}")


In [5]:
docs = loader.load()

In [6]:
docs[0].metadata

{'text': "We follow Baron's (2008) seminal work in defining the relevant terms here.Affect represents the emotions, moods, and passions of the entrepreneurit is the realized subjective feelings of (dis)pleasure attached to an object (e.g., experience; physical thing), where such feelings that can differ in length, intensity, antecedent, and so on.The entrepreneurial process consists of creatively identifying an opportunity and acting to exploit it, all under uncertainty (where the opportunity can be identified ex post as a market imperfection believed ex ante to indicate that unrealizable value existed from which some party can profit).That process is often characterized by multiple stages, non-routineness, and imperfection.Because such characterizations are associated with higher levels of emotion (e.g., where the personal stakes and pressures are greater -Cardon et al., 2012), the entrepreneurial process requires attention be paid to affect.Baron (2008) highlights the many potential 

In [7]:
from langchain_text_splitters import CharacterTextSplitter
from models import Chunk

In [8]:
chunks: list[Chunk] = []
for doc in docs:
    text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
        encoding_name="cl100k_base", chunk_size=1500, chunk_overlap=300
    )
    texts = text_splitter.split_text(doc.page_content)
    for i, text in enumerate(texts):
        chunk = Chunk(
            text=text,
            metadata=doc.metadata
        )
        chunks.append(chunk)

chunks


[Chunk(text="We follow Baron's (2008) seminal work in defining the relevant terms here.Affect represents the emotions, moods, and passions of the entrepreneurit is the realized subjective feelings of (dis)pleasure attached to an object (e.g., experience; physical thing), where such feelings that can differ in length, intensity, antecedent, and so on.The entrepreneurial process consists of creatively identifying an opportunity and acting to exploit it, all under uncertainty (where the opportunity can be identified ex post as a market imperfection believed ex ante to indicate that unrealizable value existed from which some party can profit).That process is often characterized by multiple stages, non-routineness, and imperfection.Because such characterizations are associated with higher levels of emotion (e.g., where the personal stakes and pressures are greater -Cardon et al., 2012), the entrepreneurial process requires attention be paid to affect.Baron (2008) highlights the many potenti

In [9]:
import voyageai
from dotenv import load_dotenv
from typing import List

load_dotenv()

def get_voyage_client():

    return voyageai.Client()

In [10]:
vo = get_voyage_client()

In [11]:
from models import Embedding 

embeddingsList: List[Embedding] = []
inputs: List[str] = []
for chunk in chunks:
    inputs.append(chunk.text)


# Get embeddings from Voyage AI
raw_embeddings = vo.embed(
    texts=inputs, model="voyage-4-lite", input_type="document"
)

for i, raw_embedding in enumerate(raw_embeddings.embeddings):
    embedding = Embedding(
        chunk=chunks[i],
        embedding=raw_embedding
    )
    embeddingsList.append(embedding)


In [12]:
from langchain_voyageai import VoyageAIEmbeddings
import os


embeddings = VoyageAIEmbeddings(
    voyage_api_key=os.getenv("VOYAGE_API_KEY"), model="voyage-4-lite"
)


In [13]:
from pinecone import Pinecone
from dotenv import load_dotenv



load_dotenv()

def get_pinecone_client():
    return Pinecone()

pc = get_pinecone_client()

In [14]:
from pinecone import ServerlessSpec

index_name = "langchain-test-index"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1024,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [15]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings
)


In [16]:
from uuid import uuid4

from langchain_core.documents import Document

documents: list[Document] = []

for chunk in chunks:
    documents.append(Document(page_content=chunk.text, metadata=chunk.metadata))
uuids = [str(uuid4()) for _ in range(len(documents))]
vector_store.add_documents(documents=documents, ids=uuids)


['914120df-db5f-45f6-805f-e22378bfa78d',
 'ba6073de-5961-4332-a201-7f44eab16252',
 '4ff00a47-9b55-4807-bd36-a6a69dbb4d77',
 '9ed35ee6-f4b8-4e53-8397-ffac7315e9fc',
 'c90e8e4e-f910-401c-a17d-f3fe7e043ee5',
 '53408305-2369-428f-b965-9ebc3ccc3a68',
 'ed6b6c14-16f5-4011-bd6d-81cebad48010',
 '6e5da5b4-6ebd-4959-9fee-810c4dfed85f',
 '5136f96f-0971-41a1-9bd0-ce8944d08c15',
 '1be0f6f3-88be-4b4d-b16d-a01adbda53f5',
 'c987d041-cd54-411d-98cd-82b9120192c9',
 '0e232cd8-d918-403a-8327-0d0a03edb974',
 '74ef587b-df90-498a-ae2e-1e6ed667baad',
 'a399b50d-7e2d-4a55-b4ef-7ee582c16202']

In [17]:
retriever = vector_store.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 3, "score_threshold": 0.2}
    )

In [18]:
retriever.invoke("Hope is the emotion that drives the teams, and makes them decide if they will quit or not quit")

[Document(id='6c992522-76af-438a-a765-43b4eeb9e1cb', metadata={'bboxes': "[[{'page': '3', 'x': '50.97', 'y': '294.11', 'h': '454.29', 'w': '7.17'}, {'page': '3', 'x': '39.01', 'y': '304.54', 'h': '68.99', 'w': '7.17'}], [{'page': '3', 'x': '109.59', 'y': '304.54', 'h': '395.71', 'w': '7.17'}, {'page': '3', 'x': '39.00', 'y': '315.03', 'h': '214.64', 'w': '7.17'}], [{'page': '3', 'x': '256.48', 'y': '315.03', 'h': '248.78', 'w': '7.17'}, {'page': '3', 'x': '39.00', 'y': '325.52', 'h': '466.22', 'w': '7.17'}, {'page': '3', 'x': '39.00', 'y': '335.95', 'h': '145.21', 'w': '7.17'}], [{'page': '3', 'x': '186.40', 'y': '335.31', 'h': '318.90', 'w': '7.97'}, {'page': '3', 'x': '39.00', 'y': '346.44', 'h': '88.39', 'w': '7.17'}], [{'page': '3', 'x': '129.82', 'y': '346.44', 'h': '328.80', 'w': '7.17'}], [{'page': '3', 'x': '461.14', 'y': '346.44', 'h': '44.14', 'w': '7.17'}, {'page': '3', 'x': '39.00', 'y': '356.31', 'h': '466.28', 'w': '7.73'}, {'page': '3', 'x': '39.00', 'y': '366.80', 'h': 

In [19]:
response = retriever.invoke("Hope is the emotion that drives the teams, and makes them decide if they will quit or not quit")

In [ ]:
response

[Document(id='6c992522-76af-438a-a765-43b4eeb9e1cb', metadata={'bboxes': "[[{'page': '3', 'x': '50.97', 'y': '294.11', 'h': '454.29', 'w': '7.17'}, {'page': '3', 'x': '39.01', 'y': '304.54', 'h': '68.99', 'w': '7.17'}], [{'page': '3', 'x': '109.59', 'y': '304.54', 'h': '395.71', 'w': '7.17'}, {'page': '3', 'x': '39.00', 'y': '315.03', 'h': '214.64', 'w': '7.17'}], [{'page': '3', 'x': '256.48', 'y': '315.03', 'h': '248.78', 'w': '7.17'}, {'page': '3', 'x': '39.00', 'y': '325.52', 'h': '466.22', 'w': '7.17'}, {'page': '3', 'x': '39.00', 'y': '335.95', 'h': '145.21', 'w': '7.17'}], [{'page': '3', 'x': '186.40', 'y': '335.31', 'h': '318.90', 'w': '7.97'}, {'page': '3', 'x': '39.00', 'y': '346.44', 'h': '88.39', 'w': '7.17'}], [{'page': '3', 'x': '129.82', 'y': '346.44', 'h': '328.80', 'w': '7.17'}], [{'page': '3', 'x': '461.14', 'y': '346.44', 'h': '44.14', 'w': '7.17'}, {'page': '3', 'x': '39.00', 'y': '356.31', 'h': '466.28', 'w': '7.73'}, {'page': '3', 'x': '39.00', 'y': '366.80', 'h': 

# Claim-Based Citation Chain

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

from langchain_core.tools import tool

retrieve_documents_for_claim_schema = {
    "type": "object",
    "properties": {
        "claim": {"type": "string"},
        "k": {"type": "number"},
    },
    "required": ["claim", "k"]
}

@tool(description="Retrieve top k documents for a given claim from the vector store", args_schema=retrieve_documents_for_claim_schema, response_format="content_and_artifact")
def retrieve_documents_for_claim(claim: str, k: int) -> :
    """Retrieve documents for a given claim."""
    retriever = vector_store.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": k, "score_threshold": 0.5}
    )
    response = retriever.invoke(claim)
    ## remove 
    for doc in response:
        doc.metadata.pop("bboxes", None)
        doc.metadata["claim"] = claim
    return tuple["RetrievedDocumentsForClaim", response]

tools = [retrieve_documents_for_claim]  

retrieve_document_schema = {
    "type": "object",
    "description": "A list of documents retrieved from the vector store",
    "properties": {
        "documents": {
            "type": "array",
            "items": {"type": "object", "properties": {"document": {"type": "string"}}}
        }
    }
}

retriever_system_prompt = """You are a research assistant. Given a paragragh, disect the paragraph into claims and use the retrieve_documents_for_claim tool to find 1 supporting document for EACH claim. 
    Call the tool once per claim. Return all retrieved documents. Do not make up any claims, only use the ones that are stated in the paragraph"""

model = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0, max_tokens=1000, timeout=60)

agent = create_agent(
    model,
    tools,
    response_format=ToolStrategy(schema=retrieve_document_schema),
    system_prompt=retriever_system_prompt
)

test_paragraph = """Hope is a critical fuel in startups because it sustains founders and teams through uncertainty, setbacks, and long periods without visible rewards. In the early stages, when resources are limited, product–market fit is unclear, and rejection is frequent, hope keeps people committed to the vision and willing to experiment, iterate, and improve. It transforms failure into feedback, encourages resilience after investor rejections or product pivots, and helps teams inspire employees, customers, and partners to believe in a future that does not yet exist. Without hope, the inevitable challenges of building something new can feel overwhelming; with it, obstacles become temporary and progress feels possible. """

result = agent.invoke({"messages": [
    {"role": "user", "content": f"""Paragraph: {test_paragraph}"""}
]})

for msg in result["messages"]:
    print(f"\n--- {msg.__class__.__name__} ---")
    print(msg.content if hasattr(msg, "content") else msg)

ValueError: Since response_format='content_and_artifact' a two-tuple of the message content and raw tool output is expected. Instead, generated response is of type: <class 'list'>.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate

model = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0, max_tokens=1000, timeout=60)
agent = create_agent(model, tools)

In [22]:
result
for msg in result["messages"]:
    if msg.type == "tool":
        msg.content